In [30]:
import logging
import numpy as np
from tqdm import tqdm
from tmu.models.classification.vanilla_classifier import TMClassifier
from tmu.tools import BenchmarkTimer
import random
from directories import Dicrectories
from tools import Tools
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from scipy.sparse import lil_matrix
_LOGGER = logging.getLogger(__name__)
        
def generate_knowledge(number_of_documents,number_of_features):
    #each document is set of features and has class either 0 or 1
    #context of document has vote or in case of imdb a user review is either +ve or -ve
    #how to build the document from knowledge
    #for each tw get its +ve and -ve clauses
    #from the +ve clauses take the most heigh and then get thier features. This is a class 1 document
    #from the selected features get also the most heigh clauses
    #start accumlating the features to form a document and store them along target value 1 if +ve and 0 if -ve
    #take part for training and part for test
    #feed the knowledge to the classifer TM
    #now I have to decide: 1-how many clauses to pick 2-how much for training and testing
    accumulation = 30
    documents_Y = []
    number_of_ta_chunks = int((number_of_features - 1) / 32 + 1)
    matrix = lil_matrix((number_of_documents, number_of_features), dtype=np.int32)
    
    file_list = Dicrectories.get_all_knowledge_files()
    documents_progress_bar = tqdm(total=number_of_documents, desc="Running Documents")
    for i in range(number_of_documents):
        X = np.ascontiguousarray(np.zeros(number_of_ta_chunks, dtype=np.uint32))

        while True:
            random_file = random.choice(file_list)
            if Dicrectories.check_pkl_exist(random_file,Dicrectories.knowledge):
                break
        tw_knowledge_path = Dicrectories.get_knowledge_file(random_file)
        tw_all_clauses = Tools.read_pickle_data(tw_knowledge_path)

        target_value = random.randint(0, 1)
        if target_value == 1:
            tw_filtered_clauses = [clause for clause in tw_all_clauses if clause[0] > 0]
        else:
            tw_filtered_clauses = [clause for clause in tw_all_clauses if clause[0] < 0]

        tw_clauses_subset = random.sample(tw_filtered_clauses, accumulation)
        for tw_clause in tw_clauses_subset:
            related_literals = tw_clause[1]
            for literal in related_literals:
                literal_knowledge_path = Dicrectories.knowledge_pkl_path_by_id(literal)
                if Dicrectories.check_pkl_exist(literal_knowledge_path):
                    literal_all_clauses = Tools.read_pickle_data(literal_knowledge_path)
                    if target_value == 1:
                        literal_filtered_clauses = [clause for clause in literal_all_clauses if clause[0] > 0]
                    else:
                        literal_filtered_clauses = [clause for clause in literal_all_clauses if clause[0] < 0]
                    
                    literal_clauses_subset = random.sample(literal_filtered_clauses, accumulation)
                    for literal_clause in literal_clauses_subset:
                        literals = literal_clause[1]
                        for literal in literals:
                            matrix[i, literal] += 1

        documents_Y.append(target_value)
        documents_progress_bar.update(1)
    documents_progress_bar.close()  
    return matrix.tocsr(), documents_Y

def get_knowledge():
    number_of_documents = 5
    number_of_features = 20000

    X_train, train_y = generate_knowledge(number_of_documents,number_of_features)
    X_test, test_y = generate_knowledge(number_of_documents,number_of_features)

    Y_train = np.array(train_y, dtype=np.uint32)
    Y_test = np.array(test_y, dtype=np.uint32)

    SKB = SelectKBest(chi2, k=5000)
    SKB.fit(X_train, Y_train)
    X_train = SKB.transform(X_train).toarray()
    X_test = SKB.transform(X_test).toarray()

    return X_train,Y_train,X_test,Y_test


if __name__ == "__main__":
    num_clauses = 10000
    T = 8000
    s = 2.0
    device = "GPU"
    weighted_clauses = True
    epochs = 40
    clause_drop_p = 0.75
    max_ngram = 2
    features = 5000
    imdb_num_words = 5000
    imdb_index_from = 2

    # X_train, Y_train, X_test, Y_test = get_imdb(_LOGGER, args)
    X_train, Y_train, X_test, Y_test = get_knowledge()

    print("Selecting Features.... Done!")

    tm = TMClassifier(num_clauses, T, s, platform=device, weighted_clauses=weighted_clauses,clause_drop_p=clause_drop_p)

    print(f"Running {TMClassifier} for {epochs}")
    for epoch in range(epochs):
        benchmark1 = BenchmarkTimer(logger=_LOGGER, text="Training Time")
        with benchmark1:
            tm.fit(X_train, Y_train)

        benchmark2 = BenchmarkTimer(logger=_LOGGER, text="Testing Time")
        with benchmark2:
            result = 100 * (tm.predict(X_test) == Y_test).mean()

        print(f"Epoch: {epoch + 1}, Accuracy: {result:.2f}, Training Time: {benchmark1.elapsed():.2f}s, "
                     f"Testing Time: {benchmark2.elapsed():.2f}s")



Running Documents: 100%|██████████| 5/5 [00:02<00:00,  1.98it/s]

Selecting Features.... Done!
Running <class 'tmu.models.classification.vanilla_classifier.TMClassifier'> for 40
2024-02-16 13:24:08,524 - tmu.models.base - WARNING - CUDA not installed, using CPU clause bank


2024-02-16 13:24:09,441 - __main__ - DEBUG - [0.918045ms] - Training Time
2024-02-16 13:24:11,260 - __main__ - DEBUG - [1.817174ms] - Testing Time
Epoch: 1, Accuracy: 60.00, Training Time: 0.92s, Testing Time: 1.82s
2024-02-16 13:24:11,794 - __main__ - DEBUG - [0.531681ms] - Training Time
2024-02-16 13:24:13,857 - __main__ - DEBUG - [2.060988ms] - Testing Time
Epoch: 2, Accuracy: 60.00, Training Time: 0.53s, Testing Time: 2.06s
2024-02-16 13:24:14,380 - __main__ - DEBUG - [0.521880ms] - Training Time
2024-02-16 13:24:16,702 - __main__ - DEBUG - [2.320981ms] - Testing Time
Epoch: 3, Accuracy: 60.00, Training Time: 0.52s, Testing Time: 2.32s
2024-02-16 13:24:17,262 - __main__ - DEBUG - [0.556241ms] - Training Time
2024-02-16 13:24:19,799 - __main__ - DEBUG - [2.534190ms] - Testing Time
Epoch: 4, Accuracy: 60.00, Training Time: 0.56s, Testing Time: 2.53s
2024-02-16 13:24:20,245 - __main__ - DEBUG - [0.445547ms] - Training Time
2024-02-16 13:24:22,723 - __main__ - DEBUG - [2.476743ms] - Te

KeyboardInterrupt: 